In [ ]:
print("SignGuard AI!")

In [ ]:
# ------------------
# Import Libraries
# ------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
from torchvision import datasets

from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, json, math, time, random

# seed => har safar ishga tushirganda bir hil natija chiqishi uchun
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# grafiklar va natijalar shu papkaga saqlanadi
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ------------------
# Define Dataset Path
# ------------------
# Kaggle'da dataset o'ng paneldagi "Add Input" orqali ulanadi va
# /kaggle/input/... ga read-only qilib mount bo'ladi. Yuklash, unzip yo'q.
DATASET_PATH = '/kaggle/input/gtsrb-german-traffic-sign'

print('Dataset_path:', DATASET_PATH)
print(os.listdir(DATASET_PATH))   # Train, Test, Meta, Train.csv, Test.csv, Meta.csv

In [ ]:
# ------------------
# Define Classes
# ------------------
# 43 ta class. Tartib Kaggle CSV'dagi ClassId bilan bir xil bo'lishi SHART -
# aks holda model boshqa belgini o'rganadi.
CLASSES = [
    'Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)',
    'Speed limit (60km/h)', 'Speed limit (70km/h)', 'Speed limit (80km/h)',
    'End of speed limit (80km/h)', 'Speed limit (100km/h)', 'Speed limit (120km/h)',
    'No passing', 'No passing for vehicles over 3.5t',
    'Right-of-way at next intersection', 'Priority road', 'Yield', 'Stop',
    'No vehicles', 'Vehicles over 3.5t prohibited', 'No entry', 'General caution',
    'Dangerous curve to the left', 'Dangerous curve to the right', 'Double curve',
    'Bumpy road', 'Slippery road', 'Road narrows on the right', 'Road work',
    'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
    'Beware of ice/snow', 'Wild animals crossing',
    'End of all speed and passing limits', 'Turn right ahead', 'Turn left ahead',
    'Ahead only', 'Go straight or right', 'Go straight or left', 'Keep right',
    'Keep left', 'Roundabout mandatory', 'End of no passing',
    'End of no passing by vehicles over 3.5t',
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES)

IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Compose => step by step ishlarni bajarish
# Resize  => mobilenet 224x224 kutadi
# ToTensor=> raqamga o'giramiz, H,W,C -> C,H,W, qiymatlar 0.0-1.0
# Normalize => ImageNet statistikasi, pretrained model shunga o'rgatilgan
#
# DIQQAT: flip (aylantirish) YO'Q. Ovqatni aylantirsang baribir o'sha ovqat,
# lekin belgini aylantirsang ma'nosi o'zgaradi: "chapga burilish" -> "o'ngga".
# Flip qo'shsak modelga noto'g'ri javob o'rgatgan bo'lamiz.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# val/test uchun augmentation yo'q - toza o'lchov kerak
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# normalize'siz - keyinroq stiker hujumida rasm ustiga chizamiz
raw_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
# ------------------
# Custom Dataset Class
# ------------------
# Kaggle versiyasida Train.csv va Test.csv bor: ClassId + Path ustunlari.
# ImageFolder ishlatmaymiz - papka nomlari "0","1","10","11"... bo'lib alifbo
# tartibida saralanadi va class raqamlari aralashib ketadi. CSV ishonchli.
class GTSRBDataset(Dataset):
    def __init__(self, root, csv_name, transform=None):
        df = pd.read_csv(os.path.join(root, csv_name))
        self.root = root
        self.transform = transform
        self.paths  = df['Path'].tolist()
        self.labels = df['ClassId'].astype(int).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.paths[idx])
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ------------------
# Gather and Split Data
# ------------------
# Bitta Train.csv'dan ikkita dataset obyekti yasaymiz: biri augmentation bilan
# (train uchun), ikkinchisi augmentation'siz (validation uchun). Fayllar bir xil,
# faqat transform boshqa - shunda validation rasmlari hech qachon buzilmaydi.
train_full_aug  = GTSRBDataset(DATASET_PATH, 'Train.csv', train_transform)
train_full_eval = GTSRBDataset(DATASET_PATH, 'Train.csv', val_transform)

# test - GTSRB'ning rasmiy test to'plami. Boshqa kunlarda, boshqa yo'llarda
# suratga olingan, shuning uchun test accuracy haqiqiy natija hisoblanadi.
test_dataset = GTSRBDataset(DATASET_PATH, 'Test.csv', val_transform)
test_raw     = GTSRBDataset(DATASET_PATH, 'Test.csv', raw_transform)

# 85 / 15 ga ajratamiz
n = len(train_full_aug)
perm = np.random.RandomState(SEED).permutation(n)
split = int(0.15 * n)
val_idx, train_idx = perm[:split], perm[split:]

train_dataset = Subset(train_full_aug,  train_idx)
val_dataset   = Subset(train_full_eval, val_idx)

print(f'train {len(train_dataset)}  val {len(val_dataset)}  test {len(test_dataset)}')

In [ ]:
# batch_size => birdaniga 64 ta rasmni ko'radi
# shuffle    => ketma-ketlikni yodlab olishiga yo'l qo'ymaydi
# num_workers=> backgroundda 2 ta ishchi rasm tayyorlaydi
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print('batches/epoch:', len(train_loader))

In [ ]:
# ------------------
# Datani ko'rib olamiz
# ------------------
def denorm(t):
    x = t.clone()
    for c in range(3):
        x[c] = x[c] * STD[c] + MEAN[c]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, img, y in zip(axes.ravel(), imgs, labels):
    ax.imshow(denorm(img))
    ax.set_title(CLASSES[y][:22], fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

# class balansi - GTSRB juda nomutanosib, ba'zi belgilar 10 barobar kam uchraydi.
# Buni bilib qo'yish kerak: keyinroq model aynan shu kam uchraydigan classlarda
# ishonchsiz bo'lib chiqadi.
counts = np.bincount(np.array(train_full_aug.labels), minlength=NUM_CLASSES)
order = np.argsort(counts)
plt.figure(figsize=(13, 4))
plt.bar(range(NUM_CLASSES), counts[order], color='#4C78A8')
plt.xticks(range(NUM_CLASSES), [CLASSES[i][:18] for i in order], rotation=90, fontsize=7)
plt.ylabel('training images')
plt.title(f'Class imbalance: {counts.min()} ... {counts.max()} ({counts.max()/counts.min():.1f}x)')
plt.tight_layout(); plt.show()

In [ ]:
# ------------------
# Model
# ------------------
# pretrained | lightweight | CNN
# IMAGENET1K_V1 => 1000 ta class'da o'rgatilgan model
model = mobilenet_v2(weights="IMAGENET1K_V1")

# backbone => mobilenet'ning "miyyasini" muzlatamiz, faqat oxirgi qatlamni o'rgatamiz
for p in model.features.parameters():
    p.requires_grad = False

# fine-tuning => oxiriga o'zimizning 43 ta classni qo'shamiz
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable {trainable:,} / {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = nn.CrossEntropyLoss()        # multi-class classification uchun loss
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
torch.backends.cudnn.benchmark = True    # gpu uchun eng tez algoritmni topadi

In [ ]:
# ------------------
# Training Loop
# ------------------
NUM_EPOCHS  = 8
UNFREEZE_AT = 5   # 5-epochdan backbone'ning oxirgi bloklarini ham o'rgatamiz
best_accuracy = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):

    # Yo'l belgilari ImageNet rasmlariga umuman o'xshamaydi. Faqat muzlatilgan
    # feature'lar bilan model yetarli aniqlikka chiqmaydi, shuning uchun head
    # joylashib olgach oxirgi bloklarni ham ochamiz - lekin 10 barobar kichik lr bilan,
    # aks holda tayyor bilimni buzib yuboradi.
    if epoch == UNFREEZE_AT - 1:
        for p in model.features[-4:].parameters():
            p.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.features[-4:].parameters(), 'lr': 1e-4},
            {'params': model.classifier.parameters(),    'lr': 3e-4},
        ])
        print('>>> fine-tuning boshlandi')

    model.train()
    running_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()              # gradientlarni nolga tushiramiz
        outputs = model(images)            # forward pass
        loss = criterion(outputs, labels)  # to'g'ri va noto'g'ri javob orasidagi farq
        loss.backward()                    # backpropagation
        optimizer.step()                   # weight'lar yangilanadi
        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)

    train_loss, train_acc = running_loss / total, correct / total

    # Validation => har epochdan keyin tekshiramiz
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_loss    += criterion(outputs, labels).item() * labels.size(0)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total   += labels.size(0)
    val_loss, val_acc = v_loss / v_total, v_correct / v_total

    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss);     history['val_acc'].append(val_acc)

    flag = ''
    if val_acc > best_accuracy:              # eng zo'r natijani saqlaymiz
        best_accuracy = val_acc
        torch.save(model.state_dict(), 'best.pth')
        flag = '  <- saved'

    print(f'[{epoch+1}/{NUM_EPOCHS}] train {train_loss:.4f}/{train_acc:.4f}  '
          f'val {val_loss:.4f}/{val_acc:.4f}  {time.time()-t0:.0f}s{flag}')

model.load_state_dict(torch.load('best.pth', map_location=device))
print('best val accuracy:', round(best_accuracy, 4))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
e = range(1, NUM_EPOCHS + 1)
a1.plot(e, history['train_loss'], label='train'); a1.plot(e, history['val_loss'], label='val')
a1.axvline(UNFREEZE_AT - 0.5, ls='--', c='grey', lw=1)
a1.set_title('Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.plot(e, history['train_acc'], label='train'); a2.plot(e, history['val_acc'], label='val')
a2.axvline(UNFREEZE_AT - 0.5, ls='--', c='grey', lw=1)
a2.set_title('Accuracy (--- fine-tuning)'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=130); plt.show()

In [ ]:
# ------------------
# Test Evaluation
# ------------------
# Logitlarni bir marta hisoblab eslab qolamiz. Part B'dagi hamma tahlil shu
# tensordan foydalanadi - modelni qayta-qayta ishlatish shart bo'lmaydi.
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        L.append(model(x.to(device)).cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y)

test_logits, test_labels = collect_logits(model, test_loader)
val_logits,  val_labels  = collect_logits(model, val_loader)

test_acc = (test_logits.argmax(1) == test_labels).float().mean().item()
print(f'TEST ACCURACY: {test_acc*100:.2f}%  ({len(test_labels)} ta rasm)')

In [ ]:
from sklearn.metrics import confusion_matrix

pred = test_logits.argmax(1).numpy()
true = test_labels.numpy()
cm = confusion_matrix(true, pred, labels=range(NUM_CLASSES))

plt.figure(figsize=(9, 8))
plt.imshow(np.log1p(cm), cmap='Blues'); plt.colorbar(label='log(1+count)')
plt.xlabel('predicted'); plt.ylabel('true')
plt.title(f'Confusion matrix - test accuracy {test_acc*100:.2f}%')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=130); plt.show()

# qaysi belgilar bir-biri bilan chalkashayotganini ko'ramiz
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:8]
print('Eng ko\'p chalkashuvlar:')
for t, p in pairs:
    if off[t, p] == 0: break
    print(f'  {off[t,p]:>4}x  {CLASSES[t][:30]:<32} -> {CLASSES[p][:30]}')

In [ ]:
torch.save({
    'state_dict': model.state_dict(),
    'classes': CLASSES, 'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD,
    'test_acc': test_acc,
}, 'signguard_mobilenetv2.pth')
print('saved signguard_mobilenetv2.pth')

# **PART B — Safety Audit**

Part A bitta raqam berdi: test accuracy. Shu raqam uchta xavfni yashiradi:

| Muammo | Mashinada nima bo'ladi |
|---|---|
| Ortiqcha ishonch | Model 97% deydi va xato qiladi. Hech kim buni bilmaydi. |
| "Bilmayman" yo'q | Reklama bannerini ham 43 ta belgidan biriga tiqadi. |
| Jim qulash | STOP'ga stiker yopishtirilsa javob o'zgaradi, ishonch esa baland qoladi. |

Quyida shu uchtasi o'lchanadi va bitta qarorga aylantiriladi.

In [ ]:
# ------------------
# Calibration - 95% ishonch haqiqatan 95% to'g'rimi?
# ------------------
# ECE => bashoratlarni ishonch darajasi bo'yicha guruhlaymiz va har guruhda
# "ishonch" bilan "haqiqiy aniqlik" orasidagi farqni o'lchaymiz.
def ece_and_bins(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    correct = pred.eq(labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i].item(), edges[i + 1].item()
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            rows.append((lo, hi, 0, float('nan'), float('nan'))); continue
        acc_b, conf_b = correct[m].mean().item(), conf[m].mean().item()
        ece += m.float().mean().item() * abs(acc_b - conf_b)
        rows.append((lo, hi, int(m.sum()), acc_b, conf_b))
    return ece, rows

def reliability_plot(rows, title, ax):
    centers = [(lo + hi) / 2 for lo, hi, *_ in rows]
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1, label='ideal')
    ax.bar(centers, [r[3] for r in rows], width=0.9 / len(rows),
           color='#4C78A8', edgecolor='white', label='accuracy')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    ax.set_title(title); ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=.25)

probs_raw = F.softmax(test_logits, dim=1)
ece_raw, rows_raw = ece_and_bins(probs_raw, test_labels)
print(f'ECE kalibratsiyadan oldin: {ece_raw*100:.2f}%')
print(f'o\'rtacha ishonch {probs_raw.max(1).values.mean():.4f}  |  haqiqiy aniqlik {test_acc:.4f}')

In [ ]:
# ------------------
# Temperature Scaling
# ------------------
# Bitta T soni logitlarni softmax'dan oldin bo'ladi. Bashoratni O'ZGARTIRMAYDI
# (argmax o'sha-o'sha, accuracy bir xil qoladi), faqat model qanchalik ishonchli
# "gapirishini" to'g'rilaydi.
# T validation'da topiladi, test'da emas - test'da topsak bu aldash bo'lardi.
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))   # T = exp(log_t) > 0
    @property
    def T(self):
        return self.log_t.exp()
    def forward(self, logits):
        return logits / self.T

def fit_temperature(logits, labels):
    scaler = TemperatureScaler()
    nll = nn.CrossEntropyLoss()
    opt = optim.LBFGS([scaler.log_t], lr=0.05, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nll(scaler(logits), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return scaler.T.detach().item()

TEMPERATURE = fit_temperature(val_logits, val_labels)
print(f'topilgan temperature T = {TEMPERATURE:.3f}')
print('  T > 1 => model ortiqcha ishongan, sovitamiz')
print('  T < 1 => model kam ishongan, o\'tkirlashtiramiz')

probs_cal = F.softmax(test_logits / TEMPERATURE, dim=1)
ece_cal, rows_cal = ece_and_bins(probs_cal, test_labels)
print(f'\nECE  {ece_raw*100:.2f}%  ->  {ece_cal*100:.2f}%')
print(f'accuracy o\'zgarmadi: {(test_logits.argmax(1)==test_labels).float().mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
reliability_plot(rows_raw, f'Oldin (ECE {ece_raw*100:.2f}%)', axes[0])
reliability_plot(rows_cal, f'Keyin (ECE {ece_cal*100:.2f}%)', axes[1])
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/calibration.png', dpi=130); plt.show()

In [ ]:
# ------------------
# Sticker Attack
# ------------------
# Belgi yuziga yopishtirilgan lenta/stikerning eng oddiy ko'rinishi:
# rasm ustiga bir nechta to'q rangli to'rtburchak qo'yamiz.
normalize = transforms.Normalize(mean=MEAN, std=STD)
PALETTE = np.array([[1., 1., 1.], [0., 0., 0.], [0.90, 0.85, 0.10],
                    [0.10, 0.20, 0.80], [0.85, 0.15, 0.15]], dtype=np.float32)

def apply_stickers(img, frac, rng, n_patches=2, return_cover=False):
    # to'rtburchaklar bir-birining ustiga tushishi va chetda kesilishi mumkin,
    # shuning uchun haqiqatan yopilgan maydonni o'lchab qaytaramiz
    if frac <= 0:
        return (img, 0.0) if return_cover else img
    c, h, w = img.shape
    out = img.clone()
    mask = torch.zeros(h, w, dtype=torch.bool)
    area_each = frac * h * w / n_patches
    for _ in range(n_patches):
        ar = rng.uniform(0.4, 2.5)
        ph = max(1, min(int(round(math.sqrt(area_each * ar))), h))
        pw = max(1, min(int(round(math.sqrt(area_each / ar))), w))
        top  = int(rng.integers(0, h - ph + 1))
        left = int(rng.integers(0, w - pw + 1))
        out[:, top:top+ph, left:left+pw] = torch.from_numpy(
            PALETTE[rng.integers(len(PALETTE))]).view(3, 1, 1)
        mask[top:top+ph, left:left+pw] = True
    return (out, mask.float().mean().item()) if return_cover else out

@torch.no_grad()
def occluded_logits(frac, idxs, seed=0, batch=64):
    rng = np.random.default_rng(seed)
    model.eval()
    L, Y, buf, ys, covers = [], [], [], [], []
    for i in idxs:
        img, y = test_raw[int(i)]
        occ_img, cov = apply_stickers(img, frac, rng, return_cover=True)
        covers.append(cov); buf.append(normalize(occ_img)); ys.append(y)
        if len(buf) == batch:
            L.append(model(torch.stack(buf).to(device)).cpu()); Y += ys; buf, ys = [], []
    if buf:
        L.append(model(torch.stack(buf).to(device)).cpu()); Y += ys
    return torch.cat(L), torch.tensor(Y), float(np.mean(covers))

N_ATTACK = 2000
attack_idx = np.random.RandomState(SEED).choice(len(test_raw), N_ATTACK, replace=False)

# hujum qanday ko'rinishini ko'ramiz
rng_demo = np.random.default_rng(7)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for col, frac in enumerate([0.0, 0.10, 0.20, 0.30, 0.40]):
    for row in range(2):
        img, _ = test_raw[int(attack_idx[row])]
        axes[row, col].imshow(apply_stickers(img, frac, rng_demo).permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'{int(frac*100)}% yopilgan', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/occlusion_examples.png', dpi=130); plt.show()

In [ ]:
# ------------------
# OOD ma'lumot - "bu umuman belgi emas"
# ------------------
# far-OOD: belgiga umuman o'xshamagan rasmlar. CIFAR-10 ni urinib ko'ramiz;
# server javob bermasa, test rasmlarini bo'laklarga bo'lib aralashtiramiz -
# rang va teksturasi o'sha, lekin ma'nosi yo'q.
N_OOD = 2000
try:
    cifar = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True,
                             transform=val_transform)
    ood_loader = DataLoader(Subset(cifar, range(N_OOD)), batch_size=64, num_workers=2)
    ood_logits, _ = collect_logits(model, ood_loader)
    OOD_NAME = 'CIFAR-10'
except Exception as err:
    print('CIFAR yuklanmadi:', err)
    print('=> aralashtirilgan rasmlardan far-OOD yasaymiz')
    @torch.no_grad()
    def scrambled_logits(idxs, block=32, seed=5, batch=64):
        rng = np.random.default_rng(seed)
        model.eval(); L, buf = [], []
        for i in idxs:
            img, _ = test_raw[int(i)]
            c, h, w = img.shape
            tiles = img.unfold(1, block, block).unfold(2, block, block)
            tiles = tiles.reshape(c, -1, block, block)
            tiles = tiles[:, torch.from_numpy(rng.permutation(tiles.shape[1]))]
            side = h // block
            out = tiles.reshape(c, side, side, block, block)
            out = out.permute(0, 1, 3, 2, 4).reshape(c, side*block, side*block)
            buf.append(normalize(out))
            if len(buf) == batch:
                L.append(model(torch.stack(buf).to(device)).cpu()); buf = []
        if buf:
            L.append(model(torch.stack(buf).to(device)).cpu())
        return torch.cat(L)
    ood_logits = scrambled_logits(attack_idx[:N_OOD])
    OOD_NAME = 'scrambled'

print('far-OOD manbasi:', OOD_NAME, tuple(ood_logits.shape))

# near-OOD: haqiqiy belgilar, lekin yuzining choragi yopilgan. Qiyin holat.
near_logits, near_labels, near_cover = occluded_logits(0.25, attack_idx, seed=11)
print(f'near-OOD: belgilarning {near_cover*100:.1f}% maydoni yopilgan')

In [ ]:
# ------------------
# OOD Detection - MSP vs Energy
# ------------------
# MSP    => eng katta softmax ehtimoli. Oddiy baseline.
# Energy => T * logsumexp(logits/T). Logitlarning nisbatini emas, KATTALIGINI
#           ishlatadi, shuning uchun odatda kuchliroq ishlaydi.
from sklearn.metrics import roc_auc_score

def msp_score(logits, T=1.0):
    return F.softmax(logits / T, dim=1).max(1).values

def energy_score(logits, T=1.0):
    return T * torch.logsumexp(logits / T, dim=1)

def auroc(id_s, ood_s):
    y = np.r_[np.ones(len(id_s)), np.zeros(len(ood_s))]
    return roc_auc_score(y, np.r_[id_s.numpy(), ood_s.numpy()])

results_ood = {}
for name, fn in [('MSP', msp_score), ('Energy', energy_score)]:
    results_ood[name] = {
        'far_auroc':  auroc(fn(test_logits, TEMPERATURE), fn(ood_logits,  TEMPERATURE)),
        'near_auroc': auroc(fn(test_logits, TEMPERATURE), fn(near_logits, TEMPERATURE)),
    }
    print(f'{name:<7} far-OOD AUROC {results_ood[name]["far_auroc"]:.3f}   '
          f'near-OOD AUROC {results_ood[name]["near_auroc"]:.3f}')
print('\n0.5 = tanga tashlash, 1.0 = mukammal ajratish')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, fn) in zip(axes, [('MSP', msp_score), ('Energy', energy_score)]):
    kw = dict(bins=60, alpha=.6, density=True)
    ax.hist(fn(test_logits, TEMPERATURE).numpy(), label='belgilar (ID)', **kw)
    ax.hist(fn(near_logits, TEMPERATURE).numpy(), label='25% yopilgan', **kw)
    ax.hist(fn(ood_logits,  TEMPERATURE).numpy(), label=f'{OOD_NAME} (far-OOD)', **kw)
    ax.set_title(f'{name} - far {results_ood[name]["far_auroc"]:.3f} / '
                 f'near {results_ood[name]["near_auroc"]:.3f}')
    ax.set_xlabel(f'{name} score'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ood_scores.png', dpi=130); plt.show()

In [ ]:
# ------------------
# Occlusion Robustness - qanchalik tez quladi?
# ------------------
# Ikkita chiziq: accuracy pasayadi (kutilgan), lekin XATO javoblardagi ishonch
# baland qoladi (kutilmagan - asl xavf shu).
FRACS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
occ = {}
for f in FRACS:
    lg, yb, cov = occluded_logits(f, attack_idx, seed=int(f * 1000) + 3)
    p = F.softmax(lg / TEMPERATURE, dim=1)
    conf, pred = p.max(1)
    ok = pred.eq(yb)
    occ[f] = {'cover': cov, 'acc': ok.float().mean().item(),
              'conf_wrong': conf[~ok].mean().item() if (~ok).any() else float('nan'),
              'logits': lg, 'labels': yb}
    print(f'{cov*100:5.1f}% yopilgan   accuracy {occ[f]["acc"]*100:5.1f}%   '
          f'xato javobdagi ishonch {occ[f]["conf_wrong"]*100:5.1f}%')

xs = [occ[f]['cover'] * 100 for f in FRACS]
ys_acc  = [occ[f]['acc'] * 100 for f in FRACS]
ys_conf = [occ[f]['conf_wrong'] * 100 for f in FRACS]
plt.figure(figsize=(8.5, 4.6))
plt.plot(xs, ys_acc,  'o-',  lw=2, color='#4C78A8', label='accuracy')
plt.plot(xs, ys_conf, 's--', lw=2, color='#E45756', label='xato javoblardagi ishonch')
plt.fill_between(xs, ys_acc, ys_conf, color='#E45756', alpha=.10)
plt.xlabel('belgining necha % i yopilgan'); plt.ylabel('%')
plt.title('Jim xatolik zonasi'); plt.ylim(0, 102); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/occlusion_curve.png', dpi=130); plt.show()

In [ ]:
# ------------------
# Decision Rule
# ------------------
# Hamma o'lchov bitta qatorga aylanadi:
#   ishonch >= THRESHOLD bo'lsa javob ber, aks holda UNKNOWN qaytar.
def coverage_accuracy(logits, labels, T, thresholds):
    p = F.softmax(logits / T, dim=1)
    conf, pred = p.max(1)
    ok = pred.eq(labels)
    cov, sel = [], []
    for t in thresholds:
        m = conf >= t
        cov.append(m.float().mean().item())
        sel.append(ok[m].float().mean().item() if m.any() else float('nan'))
    return np.array(cov), np.array(sel)

TARGET_SELECTIVE_ACC = 0.99
grid = np.linspace(0.0, 0.999, 400)
cov_v, sel_v = coverage_accuracy(val_logits, val_labels, TEMPERATURE, grid)

# Maqsadga yetadigan ENG PAST chegarani olamiz (= eng ko'p javob).
# Egri chiziq shovqinli, bitta tasodifiy kesishishga ishonmaslik uchun chegara
# o'zida ham, undan qattiqroq hamma nuqtada ham maqsadni bajarishi shart.
meets = (sel_v >= TARGET_SELECTIVE_ACC) | np.isnan(sel_v)
holds = np.logical_and.accumulate(meets[::-1])[::-1]
ok_idx = np.where(holds & ~np.isnan(sel_v))[0]
THRESHOLD = float(grid[ok_idx[0]]) if len(ok_idx) else 0.90
print(f'validation\'da tanlangan chegara: {THRESHOLD:.3f}')
print(f'  shu chegarada model validation\'ning {cov_v[ok_idx[0]]*100:.1f}% iga javob beradi')

cov_t, sel_t = coverage_accuracy(test_logits, test_labels, TEMPERATURE, grid)
i = int(np.argmin(np.abs(grid - THRESHOLD)))
plt.figure(figsize=(8, 4.6))
plt.plot(cov_t * 100, sel_t * 100, lw=2, color='#4C78A8')
plt.scatter([cov_t[i]*100], [sel_t[i]*100], s=70, zorder=5, color='#E45756')
plt.annotate(f'  {cov_t[i]*100:.1f}% javob / {sel_t[i]*100:.2f}% to\'g\'ri',
             (cov_t[i]*100, sel_t[i]*100), fontsize=9)
plt.xlabel('coverage - necha % inputga javob berdi')
plt.ylabel('selective accuracy - javoblarning necha % i to\'g\'ri')
plt.title('Javob bermaslik nima beradi'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/coverage_accuracy.png', dpi=130); plt.show()

In [ ]:
def gate(logits, labels=None, T=TEMPERATURE, thr=THRESHOLD):
    p = F.softmax(logits / T, dim=1)
    conf, pred = p.max(1)
    accepted = conf >= thr
    out = {'accept_rate': accepted.float().mean().item()}
    if labels is not None:
        ok = pred.eq(labels)
        out['selective_acc'] = ok[accepted].float().mean().item() if accepted.any() else float('nan')
        out['silent_fail']   = (accepted & ~ok).float().mean().item()
    return out

clean  = gate(test_logits, test_labels)
occ20  = gate(occ[0.20]['logits'], occ[0.20]['labels'])
farood = gate(ood_logits)

print('TOZA BELGILAR')
print(f'  javob berdi        {clean["accept_rate"]*100:5.1f}%')
print(f'  javoblar aniqligi  {clean["selective_acc"]*100:5.2f}%   (gate\'siz {test_acc*100:.2f}%)')
print(f'  jim xatolik        {clean["silent_fail"]*100:5.2f}%')
print('\n20% YOPILGAN BELGILAR')
print(f'  javob berdi        {occ20["accept_rate"]*100:5.1f}%')
print(f'  jim xatolik        {occ20["silent_fail"]*100:5.2f}%   (ishonch bilan XATO)')
print(f'\nUMUMAN BELGI EMAS ({OOD_NAME})')
print(f'  noto\'g\'ri javob    {farood["accept_rate"]*100:5.1f}%   (0% bo\'lishi kerak)')

In [ ]:
# ------------------
# Results Table - READMEga ko'chirish uchun
# ------------------
p20 = F.softmax(occ[0.20]['logits'] / TEMPERATURE, dim=1)
wrong20 = (p20.argmax(1) != occ[0.20]['labels']).float().mean().item()

table = f"""
| Metric | No gate | Confidence gate (T={TEMPERATURE:.2f}, thr={THRESHOLD:.2f}) |
|---|---|---|
| Inputs answered | 100% | {clean['accept_rate']*100:.1f}% |
| Accuracy on answers | {test_acc*100:.2f}% | **{clean['selective_acc']*100:.2f}%** |
| Calibration error (ECE) | {ece_raw*100:.2f}% | **{ece_cal*100:.2f}%** |
| Silent failure, clean signs | {(1-test_acc)*100:.2f}% | **{clean['silent_fail']*100:.2f}%** |
| Silent failure, 20% covered | {wrong20*100:.1f}% | **{occ20['silent_fail']*100:.1f}%** |
| Non-signs wrongly classified | 100% | **{farood['accept_rate']*100:.1f}%** |

OOD detection AUROC - far ({OOD_NAME}) / near (25% covered):
MSP {results_ood['MSP']['far_auroc']:.3f} / {results_ood['MSP']['near_auroc']:.3f} |
Energy {results_ood['Energy']['far_auroc']:.3f} / {results_ood['Energy']['near_auroc']:.3f}
"""
print(table)
with open(f'{OUT_DIR}/results_table.md', 'w') as f:
    f.write(table)

In [ ]:
# ------------------
# Save Final Model
# ------------------
# Weight'lar + ularni xavfsiz qiladigan ikki son (T va threshold) bitta faylda.
torch.save({
    'state_dict': model.state_dict(),
    'classes': CLASSES, 'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD,
    'temperature': TEMPERATURE, 'threshold': THRESHOLD,
    'test_acc': test_acc,
    'selective_acc': clean['selective_acc'],
    'coverage': clean['accept_rate'],
}, 'signguard_calibrated.pth')

audit = {
    'test_accuracy': round(test_acc, 4),
    'temperature': round(TEMPERATURE, 4),
    'threshold': round(THRESHOLD, 4),
    'ece_before': round(ece_raw, 4), 'ece_after': round(ece_cal, 4),
    'coverage': round(clean['accept_rate'], 4),
    'selective_accuracy': round(clean['selective_acc'], 4),
    'silent_fail_clean': round(clean['silent_fail'], 4),
    'silent_fail_occ20': round(occ20['silent_fail'], 4),
    'ood_source': OOD_NAME,
    'ood_accept_rate': round(farood['accept_rate'], 4),
    'auroc': {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in results_ood.items()},
    'occlusion_curve': {str(f): round(occ[f]['acc'], 4) for f in FRACS},
}
with open(f'{OUT_DIR}/audit.json', 'w') as f:
    json.dump(audit, f, indent=2)
print(json.dumps(audit, indent=2))

In [ ]:
# Kaggle'da fayllar /kaggle/working ga saqlanadi va o'ng paneldagi
# "Output" bo'limidan yuklab olinadi. Notebook'ni "Save Version" qilsang
# fayllar o'sha versiyada saqlanib qoladi.
!ls -la /kaggle/working
print('\nO\'ng paneldagi Output bo\'limidan signguard_calibrated.pth va outputs/ ni yuklab ol')

---
## **signguard** model usage

Yuqoridagi `signguard_calibrated.pth` faylini yuklab, bitta rasmni tekshirish.

In [ ]:
# ------------------
# Required Imports
# ------------------
import torch, torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
from PIL import Image
import matplotlib.pyplot as plt

# ------------------
# Load Model
# ------------------
ckpt = torch.load('/kaggle/working/signguard_calibrated.pth', map_location='cpu')
CLASSES     = ckpt['classes']
TEMPERATURE = ckpt['temperature']
THRESHOLD   = ckpt['threshold']

model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, len(CLASSES))
model.load_state_dict(ckpt['state_dict'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((ckpt['img_size'], ckpt['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=ckpt['mean'], std=ckpt['std']),
])

# ------------------
# Upload & Predict
# ------------------
# Kaggle'da rasm yuklash uchun o'ng paneldan "Add Input -> Upload" qilasan,
# keyin shu yerga yo'lini yozasan.
TEST_IMAGES = ['/kaggle/input/<sening-datasetingt>/sign.jpg']

for name in TEST_IMAGES:
    image = Image.open(name).convert('RGB')
    x = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x) / TEMPERATURE, dim=1)[0].cpu()
    conf, idx = probs.max(0)
    top_p, top_i = probs.topk(3)

    plt.imshow(image); plt.axis('off')
    if conf.item() >= THRESHOLD:
        plt.title(f'{CLASSES[idx]}  —  {conf.item()*100:.1f}%')
    else:
        # ishonch yetarli emas => javob bermaymiz
        plt.title(f'UNKNOWN  (eng yaqin: {CLASSES[idx]}, {conf.item()*100:.1f}%)')
    plt.show()

    for p, i in zip(top_p, top_i):
        print(f'  {p.item()*100:5.1f}%  {CLASSES[i]}')